# Lab #6: Keras MLP for Regression
**Name:** Chris Abraham Blessan &nbsp;&nbsp;|&nbsp;&nbsp; **Reg. No:** 24215207

**Objective:** Implement a Multi-Layer Perceptron (MLP) using Keras/TensorFlow for a regression
problem, and analyze the effect of different activation functions and loss functions on model
performance.

---
## 1. Dataset Description and Source

**Dataset:** California Housing Prices (built into `scikit-learn` as `fetch_california_housing`,
originally derived from the 1990 U.S. Census).

- **Task:** Predict the median house value (in units of \$100,000) for a California district —
  a **continuous target variable**, so this is a regression problem.
- **Samples:** 20,640 districts (block groups).
- **Features (8, all numeric):** `MedInc` (median income), `HouseAge`, `AveRooms`, `AveBedrms`,
  `Population`, `AveOccup`, `Latitude`, `Longitude`.
- **Why this dataset:** it is a classic, instructor-approved-style "house price prediction"
  problem, has no missing values (so preprocessing focuses on scaling), has real-world skew and
  outliers (income and house value are capped/skewed), which makes it useful for comparing
  loss functions like MSE vs. MAE vs. Huber.


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. Load and Explore the Dataset

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print("Shape:", df.shape)
df.head()


In [ ]:
df.describe()


In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Check dtypes -> confirm there are no categorical variables to encode
print("\nDtypes:")
print(df.dtypes)


In [ ]:
# Quick look at target distribution -> motivates loss-function choice later
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(df['MedHouseVal'], bins=50, color='steelblue')
ax[0].set_title('Distribution of Target (MedHouseVal)')
ax[0].set_xlabel('Median House Value ($100k)')

corr = df.corr(numeric_only=True)['MedHouseVal'].drop('MedHouseVal').sort_values()
ax[1].barh(corr.index, corr.values, color='seagreen')
ax[1].set_title('Feature Correlation with Target')
plt.tight_layout()
plt.show()


**Observation:** `MedHouseVal` is right-skewed with a spike/cap around 5.0 (\$500k) — a small
but real cluster of outlier/clipped values. `MedInc` is the strongest predictor. This skew and the
presence of capped outliers is the dataset-specific justification we will refer back to when
choosing between MSE, MAE, and Huber loss in Section 5.

## 3. Data Preprocessing

- **Missing values:** none present (confirmed above), so no imputation is required.
- **Categorical variables:** none — all 8 features are numeric, so no encoding is required.
- **Feature scaling:** MLPs are sensitive to feature scale (this affects gradient descent
  convergence, and matters even more for `sigmoid`/`tanh`, which saturate for large inputs).
  We apply `StandardScaler` (zero mean, unit variance) to the input features.
- **Target scaling:** we leave the target in its original units ($100k) so that MAE/RMSE are
  directly interpretable; loss functions are computed on this scale for all models so comparisons
  stay fair.
- **Split:** 80% train / 20% test, with a further 10% of the training split held out for
  validation during training.


In [ ]:
X = df.drop(columns=['MedHouseVal']).values
y = df['MedHouseVal'].values

# Train/test split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# Further split train -> train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1, random_state=SEED
)

# Feature scaling (fit ONLY on training data to avoid leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)


## 4. MLP Architecture

Every model shares the same base architecture so that activation/loss are the only things being
compared (as required by the assignment):

- Input layer: 8 features
- Hidden layer 1: 64 units
- Hidden layer 2: 32 units
- Output layer: 1 neuron, **linear** activation (regression output must be unbounded)

Only the **hidden-layer activation function** and the **loss function** are varied between
experiments; optimizer (Adam), learning rate, batch size, epochs, and architecture are held
constant so the comparison is fair.


In [ ]:
def build_mlp(input_dim, activation='relu', loss='mse'):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation=activation),
        layers.Dense(32, activation=activation),
        layers.Dense(1, activation='linear')  # linear output for regression
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=loss,
        metrics=['mae', keras.metrics.RootMeanSquaredError(name='rmse')]
    )
    return model

EPOCHS = 100
BATCH_SIZE = 32

def train_model(activation='relu', loss='mse', verbose=0):
    model = build_mlp(X_train.shape[1], activation=activation, loss=loss)
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True
    )
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=verbose
    )
    return model, history


## 5. Experiment 1 — Comparing Activation Functions

Loss function is **held fixed at MSE** while we compare `ReLU`, `Sigmoid`, and `Tanh` in the
hidden layers.

**Why these three, and what we expect for this dataset:**
- **ReLU** — does not saturate for positive inputs, so gradients stay large and training tends to
  converge fastest; the usual default for regression MLPs on tabular data like this.
- **Sigmoid** — squashes activations into (0, 1); on a standardized 8-feature tabular dataset like
  this it is prone to vanishing gradients through two hidden layers, so we expect slower,
  possibly worse convergence.
- **Tanh** — zero-centered version of sigmoid, (-1, 1); usually a middle ground between ReLU and
  sigmoid on tabular data — better gradient flow than sigmoid, but can still saturate for large
  standardized inputs.


In [ ]:
activations = ['relu', 'sigmoid', 'tanh']
activation_results = {}

for act in activations:
    print(f"Training with activation = {act} ...")
    model, history = train_model(activation=act, loss='mse')
    activation_results[act] = {'model': model, 'history': history}
print("Done.")


In [ ]:
plt.figure(figsize=(8, 5))
for act, res in activation_results.items():
    plt.plot(res['history'].history['loss'], label=f'{act} - train')
    plt.plot(res['history'].history['val_loss'], '--', label=f'{act} - val')
plt.title('Training vs Validation Loss (MSE) by Activation Function')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

activation_metrics = {}
for act, res in activation_results.items():
    activation_metrics[act] = evaluate_model(res['model'], X_test, y_test)

pd.DataFrame(activation_metrics).T


## 6. Experiment 2 — Comparing Loss Functions

Activation function is now **held fixed** at whichever performed best in Experiment 1 (typically
ReLU), while we compare three regression losses: **MSE**, **MAE**, and **Huber**.

**Why these three, and dataset-specific justification:**
- **MSE (Mean Squared Error)** — squares the error, so it penalizes large errors heavily. Good
  default, but sensitive to the outliers/capped values we saw in the target distribution
  (Section 2), so it can be pulled around by the small cluster of expensive districts.
- **MAE (Mean Absolute Error)** — penalizes all errors linearly, making it more robust to
  outliers than MSE. Directly interpretable in the same units as house value.
- **Huber Loss** — behaves like MSE for small errors (smooth gradient near zero) and like MAE for
  large errors (robust to outliers). This is a natural fit here because the target has both a
  well-behaved bulk of values and a skewed tail of expensive/capped districts.


In [ ]:
best_activation = min(activation_metrics, key=lambda a: activation_metrics[a]['RMSE'])
print("Best activation from Experiment 1 (lowest test RMSE):", best_activation)

losses = ['mse', 'mae', keras.losses.Huber(delta=1.0)]
loss_names = ['mse', 'mae', 'huber']
loss_results = {}

for name, loss_fn in zip(loss_names, losses):
    print(f"Training with loss = {name} ...")
    model, history = train_model(activation=best_activation, loss=loss_fn)
    loss_results[name] = {'model': model, 'history': history}
print("Done.")


In [ ]:
plt.figure(figsize=(8, 5))
for name, res in loss_results.items():
    plt.plot(res['history'].history['loss'], label=f'{name} - train')
    plt.plot(res['history'].history['val_loss'], '--', label=f'{name} - val')
plt.title(f'Training vs Validation Loss by Loss Function (activation = {best_activation})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
loss_metrics = {}
for name, res in loss_results.items():
    loss_metrics[name] = evaluate_model(res['model'], X_test, y_test)

pd.DataFrame(loss_metrics).T


## 7. Comparison Table — All Experiments

In [ ]:
rows = []
for act, m in activation_metrics.items():
    rows.append({'Experiment': f'Activation-{act}', 'Activation': act, 'Loss Function': 'mse',
                 'MAE': m['MAE'], 'RMSE': m['RMSE'], 'R2': m['R2']})
for name, m in loss_metrics.items():
    rows.append({'Experiment': f'Loss-{name}', 'Activation': best_activation, 'Loss Function': name,
                 'MAE': m['MAE'], 'RMSE': m['RMSE'], 'R2': m['R2']})

comparison_df = pd.DataFrame(rows).round(4)
comparison_df


In [ ]:
# Identify the single best overall combination (lowest test RMSE across all 6 runs)
best_row = comparison_df.loc[comparison_df['RMSE'].idxmin()]
print("Best overall combination:")
print(best_row)


## 8. Analysis and Interpretation

*(Fill in the specific numbers from your own run — the structure below tells you what to look
for and how to phrase the conclusions.)*

- **Which activation function performed best?** Compare the three rows under `Activation-*` in
  the table above by RMSE/R². On tabular data like this, ReLU typically converges fastest and
  gives the lowest error because it avoids the vanishing-gradient problem that sigmoid suffers
  from through two hidden layers.

- **Which loss function performed best?** Compare the `Loss-*` rows. Watch specifically whether
  MAE or Huber gives a lower MAE than MSE does (a sign that the outlier-heavy tail of the target
  is being handled better), while checking whether MSE still gives the lowest RMSE (since RMSE
  itself squares errors and so naturally favors the model trained to minimize squared error).

- **Which combination produced the best regression performance?** Read off `best_row` above —
  report its activation, loss, and metrics as the final selected model.

- **Did activation choice significantly affect convergence?** Look at the loss-curve plot in
  Section 5: sigmoid's curve should be visibly slower to drop and/or noisier than ReLU's/tanh's,
  which is the expected vanishing-gradient symptom.

- **Did any loss function make the model more robust to outliers?** Compare MAE metric values
  (not just the training loss) across the `Loss-*` rows — a model trained with Huber or MAE loss
  that achieves a lower test MAE than the MSE-trained model indicates improved robustness to the
  capped/outlier house values identified in Section 2.

- **Overfitting / underfitting from the learning curves:** In each loss-curve plot, a large and
  growing gap between the solid (train) and dashed (validation) lines indicates overfitting; both
  curves plateauing at a high loss value indicates underfitting. `EarlyStopping` (patience=10,
  `restore_best_weights=True`) was used in training specifically to stop each run at its best
  validation performance and mitigate overfitting.


## 9. Final Model Selection and Justification

Based on the comparison table in Section 7, the final model is the combination in `best_row`
(lowest test RMSE). This combination is justified by the dataset characteristics discussed
throughout this notebook: California Housing has no missing values or categorical features (so
preprocessing is scaling-only), the target is right-skewed with a capped/outlier tail (motivating
comparison of MSE vs. MAE vs. Huber), and it is a moderately sized tabular dataset for which
ReLU-based MLPs typically converge fastest and most reliably.
